# 第02章 逼近的智慧 · C2组人工复核

- 小组：C2组（G9）· 第02章 C 批判
- 复核日期：2026-11-10
- 复核方式：人工独立推导 + 本 Notebook 实际运行；AI 仅用于检索候选、排版与代码草稿，关键结论由组员逐项核对。

| 版本 | 仓库快照 | 文件 SHA-256（本组下载件） | 说明 |
| --- | --- | --- | --- |
| A1（G8） | `main@77d566a713915bfa770faa5375fe9a6b949506e3` | `06AA6B3C0D6259EBEC39FCC19E40DCF77D317C5F9EFA1BF2A511CCF4867403BD` | 五节稿；本表按正文 `A1-G8-正文.md` 复核 |
| A2（G21） | 同上 | `EEACE11E2E6684CA63F9394CA8B1357DA844E242772F4CE5E2766BD9421ABBDA` | 四节稿；本表按正文 `A2-G21-正文.md` 复核 |
| B1（G12） | 同上 | `D1F90AE8DBCA7FA84EA43E154C0959797FCD13504A845AD2E6D26CFE075757FC`（报告）<br>`1F2D625CA85BD19AB72E70F27C7EFBBD1E6B42DCC4B9CFBBDBB1E2322C1C8DC7`（修正清单） | B1 报告自述的 A1/A2 版本与当前 main 不完全相同，见下文 |
| B2（G22） | 同上 | 未取得 | `工作/` 下无 B2 文件或分支；本组不能假称已复核 B2 |

> 版本边界：A1 编写报告自述的 `601b2f70…` 在公共仓库不可达；B1 报告对部分成员使用本地 SHA-256 快照。本表以 2026-11-10 的 `main@77d566a7` 下载件为准，并在正文中保留 B1 自述版本，避免把不同快照混为一个。

In [1]:
import sys, platform
import numpy as np
print("Python", sys.version.split()[0])
print("NumPy", np.__version__)
print("Platform", platform.platform())
np.set_printoptions(precision=12, suppress=False)

Python 3.14.4
NumPy 2.5.4
Platform Windows-11-10.0.26200-SP0


## 2.1 插值与 Runge 现象

### 独立推导要点

对 $n+1$ 个互异节点，插值多项式 $p_n$ 的余项为

$$
f(x)-p_n(x)=\frac{f^{(n+1)}(\xi)}{(n+1)!}\omega(x),\qquad
\omega(x)=\prod_{j=0}^{n}(x-x_j).
$$

在 $[-1,1]$ 上，首一多项式的最小最大模由 $2^{-n}T_{n+1}$ 达到，故取 $T_{n+1}$ 的零点时

$$
\|\omega\|_\infty=2^{-n},\qquad
\|f-p_n\|_\infty\le \frac{\|f^{(n+1)}\|_\infty}{2^n(n+1)!}.
$$

这个界是充分条件，不能反过来证明等距节点必然发散；Runge 现象要用具体函数与节点族验证。另一个容易混淆的点是 $n$ 的含义：A1 用 $p_n$（次数 $n$，节点 $n+1$），A2 用节点数 $n$（次数 $n-1$）。两者表格不能逐行直接比较。

In [2]:
def bary_weights(x):
    x = np.asarray(x, dtype=float)
    n = x.size
    w = np.ones(n)
    for i in range(n):
        for j in range(n):
            if i != j:
                w[i] /= (x[i] - x[j])
    return w

def bary_eval(xn, yn, x):
    xn = np.asarray(xn, dtype=float); yn = np.asarray(yn, dtype=float)
    x = np.asarray(x, dtype=float); w = bary_weights(xn)
    out = np.empty_like(x)
    for k, xx in enumerate(x):
        d = xx - xn
        hit = np.where(d == 0.0)[0]
        if hit.size:
            out[k] = yn[hit[0]]
        else:
            t = w / d
            out[k] = np.dot(t, yn) / t.sum()
    return out

f_runge = lambda x: 1.0 / (1.0 + 25.0 * x * x)
z = -1.0 + 2.0 * (np.arange(6001) + 0.371) / 6001.0

print("A1 的 n 是插值多项式次数，节点数 = n+1")
for n in (8, 16, 32):
    N = n + 1
    xe = np.linspace(-1.0, 1.0, N); ye = f_runge(xe)
    xc = np.cos((2*np.arange(N)+1)*np.pi/(2*N)); yc = f_runge(xc)
    ee = np.max(np.abs(bary_eval(xe, ye, z) - f_runge(z)))
    ec = np.max(np.abs(bary_eval(xc, yc, z) - f_runge(z)))
    print(f"n={n:2d}  N={N:2d}  equispaced={ee:.10g}  cheb={ec:.10g}")

print("\nA2 的 n 是节点数，次数 = n-1")
for n in (5, 9, 11, 15, 21, 31):
    xe = np.linspace(-1.0, 1.0, n); ye = f_runge(xe)
    xc = np.cos((2*np.arange(n)+1)*np.pi/(2*n)); yc = f_runge(xc)
    ee = np.max(np.abs(bary_eval(xe, ye, z) - f_runge(z)))
    ec = np.max(np.abs(bary_eval(xc, yc, z) - f_runge(z)))
    print(f"n={n:2d}  equispaced={ee:.10g}  cheb={ec:.10g}")

print("\n节点处误差（n=9, 等距）：",
      np.max(np.abs(bary_eval(np.linspace(-1,1,9), f_runge(np.linspace(-1,1,9)), np.linspace(-1,1,9)) - f_runge(np.linspace(-1,1,9)))))

print("\nChebyshev 节点多项式的最大模（应等于 2^(1-N)）")
for N in (9, 17, 33):
    xc = np.cos((2*np.arange(N)+1)*np.pi/(2*N))
    omega = np.prod(z[:, None] - xc[None, :], axis=1)
    print(f"N={N:2d}  max|omega|={np.max(np.abs(omega)):.10e}  2^(1-N)={2.0**(1-N):.10e}")

A1 的 n 是插值多项式次数，节点数 = n+1
n= 8  N= 9  equispaced=1.045176594  cheb=0.1708355598
n=16  N=17  equispaced=14.39376327  cheb=0.03261357363
n=32  N=33  equispaced=5059.0114  cheb=0.001401742271

A2 的 n 是节点数，次数 = n-1
n= 5  equispaced=0.4383570974  cheb=0.4020168946
n= 9  equispaced=1.045176594  cheb=0.1708355598
n=11  equispaced=1.915658918  cheb=0.1091534051


n=15  equispaced=7.194845048  cheb=0.04660225894
n=21  equispaced=59.82153324  cheb=0.01533372309
n=31  equispaced=2388.282425  cheb=0.002061573239

节点处误差（n=9, 等距）： 0.0

Chebyshev 节点多项式的最大模（应等于 2^(1-N)）
N= 9  max|omega|=3.9062498273e-03  2^(1-N)=3.9062500000e-03
N=17  max|omega|=1.5258789060e-05  2^(1-N)=1.5258789062e-05
N=33  max|omega|=2.3283060686e-10  2^(1-N)=2.3283064365e-10


**读数结论。** A1 的 `n=8` 行与 A2 的 `n=9` 行给出相同的 `1.0452`，因为 A1 的 `n=8` 实际是 9 个节点；A1 的 `n=16`、`n=32` 分别对应 17、33 个节点。两组的公式与数值都自洽，但符号 `n` 的定义不同，教材必须加一列“节点数”或统一改成次数，否则学生会把两表当成同一批实验直接比较。

## 2.2 最小二乘：正规方程、换基与条件数

### 独立推导要点

$$
J(c)=\|Ac-y\|_2^2,\qquad \nabla J(c)=2A^{\mathsf T}(Ac-y),
$$

最优解满足正规方程 $A^{\mathsf T}Ac=A^{\mathsf T}y$。若 $A$ 列满秩，$A^{\mathsf T}A$ 正定，解唯一；若 $A$ 列不满秩，所有解给出相同拟合值，但系数不唯一。

若 $A_\psi=A_\phi B$ 且 $B$ 可逆，则两组基张成同一列空间，最小二乘拟合值相同；换基只改变坐标与数值条件。对 2-范数，

$$
\kappa_2(A^{\mathsf T}A)=\kappa_2(A)^2.
$$

使用 Chebyshev 多项式作 **最小二乘基**，与极小极大（一致）逼近是两个不同目标；前者最小化 $\ell_2$ 残差，后者最小化 $\ell_\infty$ 误差，解一般不同。

In [3]:
def design_power(x, d):
    return np.vander(np.asarray(x, dtype=float), d+1, increasing=True)

def design_cheb(x, d):
    return np.polynomial.chebyshev.chebvander(np.asarray(x, dtype=float), d)

# A1 的主实验：41 点，[-1,1]，e^x，噪声标准差 0.015，种子 2026，d=12
x1 = np.linspace(-1.0, 1.0, 41)
y1 = np.exp(x1) + np.random.default_rng(2026).normal(0.0, 0.015, 41)
d = 12
A1p = design_power(x1, d); A1c = design_cheb(x1, d)
cp = np.linalg.lstsq(A1p, y1, rcond=None)[0]
cc = np.linalg.lstsq(A1c, y1, rcond=None)[0]
xt1 = np.linspace(-1.0, 1.0, 2001)
pp = design_power(xt1, d) @ cp
pc = design_cheb(xt1, d) @ cc
print("A1 主实验（d=12）")
print(f"  kappa(power) = {np.linalg.cond(A1p):.10g}")
print(f"  kappa(cheb)  = {np.linalg.cond(A1c):.10g}")
print(f"  residual(power) = {np.linalg.norm(A1p@cp-y1):.10g}")
print(f"  residual(cheb)  = {np.linalg.norm(A1c@cc-y1):.10g}")
print(f"  max|power-exp| = {np.max(np.abs(pp-np.exp(xt1))):.10g}")
print(f"  max|cheb-exp|  = {np.max(np.abs(pc-np.exp(xt1))):.10g}")
print(f"  max|power-cheb| = {np.max(np.abs(pp-pc)):.10g}")
print(f"  kappa(A^T A) / kappa(A)^2 = {np.linalg.cond(A1p.T@A1p)/np.linalg.cond(A1p)**2:.12f}")

# A2 的主实验：15 点，[0,1]，e^x，噪声标准差 0.03，种子 2026
x2 = np.linspace(0.0, 1.0, 15)
y2 = np.exp(x2) + np.random.default_rng(2026).normal(0.0, 0.03, 15)
xt2 = np.linspace(0.0, 1.0, 400); yt2 = np.exp(xt2)
print("\nA2 主实验（幂基在 x∈[0,1]，Chebyshev 基在 t=2x-1）")
print("d  kappa(power)  kappa(cheb)  train_max  test_max")
for dd in (1, 2, 3, 6, 9, 12, 14):
    Ap = design_power(x2, dd)
    Ac = design_cheb(2*x2-1, dd)
    cp = np.linalg.lstsq(Ap, y2, rcond=None)[0]
    pred = design_power(xt2, dd) @ cp
    print(f"{dd:2d} {np.linalg.cond(Ap):12.4e} {np.linalg.cond(Ac):11.4e} {np.max(np.abs(Ap@cp-y2)):10.4e} {np.max(np.abs(pred-yt2)):10.4e}")

# A2 的测试网格与训练点重合数
overlap = np.sum([np.any(np.isclose(t, x2, atol=0.0, rtol=0.0)) for t in xt2])
print(f"\nA2 400 点测试网格与 15 个训练点重合数：{overlap}")
mask = np.array([not np.any(np.isclose(t, x2, atol=0.0, rtol=0.0)) for t in xt2])
for dd in (3, 6, 9, 12, 14):
    Ap = design_power(x2, dd); cp = np.linalg.lstsq(Ap, y2, rcond=None)[0]
    pred = design_power(xt2, dd) @ cp
    print(f"d={dd:2d}  test_max(all)={np.max(np.abs(pred-yt2)):.6e}  test_max(excluding overlap)={np.max(np.abs(pred[mask]-yt2[mask])):.6e}")

# A2 报告的正规方程与 lstsq 系数差
for dd in (9, 12):
    Ap = design_power(x2, dd)
    c_ls = np.linalg.lstsq(Ap, y2, rcond=None)[0]
    c_ne = np.linalg.solve(Ap.T@Ap, Ap.T@y2)
    print(f"d={dd}: max|coef(normal)-coef(lstsq)| = {np.max(np.abs(c_ne-c_ls)):.6g}")

A1 主实验（d=12）
  kappa(power) = 16674.68569
  kappa(cheb)  = 2.458870848
  residual(power) = 0.06932629866
  residual(cheb)  = 0.06932629866
  max|power-exp| = 0.01892024582
  max|cheb-exp|  = 0.01892024582
  max|power-cheb| = 9.381384558e-15
  kappa(A^T A) / kappa(A)^2 = 1.000000006807

A2 主实验（幂基在 x∈[0,1]，Chebyshev 基在 t=2x-1）
d  kappa(power)  kappa(cheb)  train_max  test_max
 1   4.1161e+00  1.6202e+00 1.3858e-01 1.3376e-01
 2   2.0291e+01  1.6972e+00 5.2383e-02 1.3655e-02
 3   1.0547e+02  1.8210e+00 5.1682e-02 2.0447e-02
 6   1.8193e+04  1.8450e+00 5.2027e-02 2.5302e-02
 9   4.6283e+06  2.9876e+00 3.4190e-02 3.1505e-02
12   2.4127e+09  1.6511e+01 5.6389e-03 1.7641e-01
14   4.0324e+11  2.2573e+02 2.5780e-08 9.6885e-01

A2 400 点测试网格与 15 个训练点重合数：8
d= 3  test_max(all)=2.044732e-02  test_max(excluding overlap)=2.009118e-02
d= 6  test_max(all)=2.530163e-02  test_max(excluding overlap)=2.530163e-02
d= 9  test_max(all)=3.150494e-02  test_max(excluding overlap)=3.150494e-02
d=12  test_max(all)=

**读数结论。** A1 与 A2 的代数结论都成立；A1 的 `kappa=1.6675e4` 与 A2 的 `4.63e6` 不是同一实验（区间、样本数、噪声、基的变量不同），不能横向比较。A2 的 400 点测试网格与 15 个训练点重合 8 个（含两端点与 6 个内点）；去掉重合点后最大测试误差略变（如 $d=3$ 从 `2.0447e-2` 到 `2.0091e-2`），过拟合结论不变，但“独立测试”应改成“含 8 个训练点的测试网格”或直接排除重合点。A2 报的系数差 `4.152` 本机复算为 `2.23`（$d=9$）；量级同为 $O(1)$，属求解器/环境相关，不能当作精确常数。

## 2.3 数值积分与自动微分

### 独立推导要点

**复合梯形。** 对 $f\in C^2[a,b]$，$h=(b-a)/N$，

$$
I-T_N=-\frac{(b-a)h^2}{12}f''(\xi).
$$

对 $f=x^2$、$[0,1]$，$I=1/3$，$T_N=1/3+1/(6N^2)$，故 $T_4=11/32$、$T_8=43/128$，误差比恰为 4。这个“恰为 4”是二次函数的代数事实，不是所有 $C^2$ 函数的逐次恒等。

**Simpson。** 设区间 $[u,v]$，全宽 $H=v-u$，中点 $m=(u+v)/2$。则

$$
\int_u^v f\approx \frac{H}{6}\bigl[f(u)+4f(m)+f(v)\bigr].
$$

若令半宽 $h=H/2$，则 $m=u+h$，公式为

$$
\int_u^v f\approx \frac{h}{3}\bigl[f(u)+4f(u+h)+f(v)\bigr].
$$

A1 正文写的是 $\frac h6[f(u)+4f(u+h)+f(v)]$：它把“全宽系数 $H/6$”和“半宽节点 $u+h$”混在一起。按半宽读，系数应为 $h/3$；按全宽读，节点应为 $u+H/2$。两种读法下原式都不对。

**中心差分。** 由 Taylor 展开，

$$
D_h^0F(x)=\frac{F(x+h)-F(x-h)}{2h}
=F'(x)+\frac{h^2}{6}F'''(x)+O(h^4),
$$

所以 $D_h^0F-F'=+\frac{h^2}{6}F'''$，等价地 $F'-D_h^0=-\frac{h^2}{6}F'''$。A1 正文式 (2.4.2) 写的是后者，正确；A1 题解 4.2 写成 $D_h^0F-F'=-\frac{h^2}{6}F'''$，把负号抄反了。

**最优步长。** 前向总误差 $\lesssim M_2h/2+2\varepsilon|f|/h$，极小点 $h\sim\sqrt\varepsilon$；中心 $\lesssim M_3h^2/6+\varepsilon|f|/h$，极小点 $h\sim\varepsilon^{1/3}$。

In [4]:
f2 = lambda x: x*x
I2 = 1.0/3.0
def trapezoid(f, a, b, N):
    x = np.linspace(a, b, N+1); y = f(x)
    return (b-a)/N * (0.5*y[0] + y[1:-1].sum() + 0.5*y[-1])

print("复合梯形：f=x^2, [0,1]")
print("N  T_N  |I-T_N|  ratio")
prev = None
for N in (4, 8, 16, 32, 64, 128):
    T = trapezoid(f2, 0.0, 1.0, N); err = abs(I2-T)
    ratio = "" if prev is None else f"{prev/err:.6f}"
    print(f"{N:4d} {T:.15f} {err:.6e} {ratio}")
    prev = err

# Simpson 的三种写法：用 x^2 在 [0,1] 上，真值 1/3
u, v = 0.0, 1.0; H = v-u; m = (u+v)/2
S_correct = H/6*(f2(u)+4*f2(m)+f2(v))
S_half_h6 = (H/2)/6*(f2(u)+4*f2(u+H/2)+f2(v))
S_full_h6_wrongnode = H/6*(f2(u)+4*f2(u+H)+f2(v))
print("\nSimpson 三种写法（f=x^2, [0,1], 真值 1/3）")
print(f"  H/6, midpoint u+H/2        = {S_correct:.15f}")
print(f"  (H/2)/6, node u+H/2        = {S_half_h6:.15f}")
print(f"  H/6, node u+H (A1 原式)    = {S_full_h6_wrongnode:.15f}")

# A1 的 h^{3/6} 排版错误
h = 0.25
print(f"\nh=0.25: h^(3/6)={h**(3/6):.12g}; h^3/6={h**3/6:.12g}; ratio={h**(3/6)/(h**3/6):.6g}")

# A1 的区间缩放：N=8, f=x^2
e1 = abs(I2 - trapezoid(f2, 0.0, 1.0, 8))
I10 = 1000.0/3.0
e10 = abs(I10 - trapezoid(f2, 0.0, 10.0, 8))
print(f"\n[0,1] N=8 error={e1:.12g}; [0,10] N=8 error={e10:.12g}; ratio={e10/e1:.6f}")

# 中心差分：F=sin(x^2)+e^x, x=0.7
F = lambda x: np.sin(x*x) + np.exp(x)
Fp = lambda x: 2*x*np.cos(x*x) + np.exp(x)
Fpp = lambda x: 2*np.cos(x*x) - 4*x*x*np.sin(x*x) + np.exp(x)
F3 = lambda x: -12*x*np.sin(x*x) - 8*x**3*np.cos(x*x) + np.exp(x)
x0 = 0.7
ref = Fp(x0)
h0 = 0.1
D0 = (F(x0+h0)-F(x0-h0))/(2*h0)
Df = (F(x0+h0)-F(x0))/h0
print(f"\nF'(0.7) = {ref:.16g}")
print(f"center h=0.1: D={D0:.16g}, D-F'={D0-ref:+.12e}, |D-F'|={abs(D0-ref):.12e}")
print(f"forward h=0.1: D={Df:.16g}, D-F'={Df-ref:+.12e}")
print(f"A1 写的 3.2563 = F' + |D-F'| = {ref + abs(D0-ref):.10f}")
print(f"h^2/6 * F'''(0.7) = {h0**2/6*F3(x0):+.12e}；与 D-F' 同号，说明正确符号是 +")

print("\nA2 的 16 档中心差分扫描（* 为 A2 表中选录的 7 档）")
selected = {1e-1,1e-2,1e-3,1e-5,1e-8,1e-12,1e-16}
for hh in [10.0**(-k) for k in range(1,17)]:
    D = (F(x0+hh)-F(x0-hh))/(2*hh)
    mark = "*" if hh in selected else " "
    print(f"{mark} h={hh:.0e}  |D-F'|={abs(D-ref):.6e}")

# 最优步长理论值与扫描最小值
M2 = np.max(np.abs(Fpp(np.linspace(0.6, 0.8, 2001))))
M3 = np.max(np.abs(F3(np.linspace(0.6, 0.8, 2001))))
eps = np.finfo(float).eps
print(f"\n|F(0.7)|={abs(F(x0)):.10g}; |F'(0.7)|={abs(ref):.10g}; M2={M2:.10g}; M3={M3:.10g}")
print(f"theory forward h* = {np.sqrt(4*eps*abs(F(x0))/M2):.6e}")
print(f"theory center  h* = {(3*eps*abs(F(x0))/M3)**(1/3):.6e}")
hs = np.array([10.0**(-k) for k in range(1,17)])
errs = np.array([abs((F(x0+h)-F(x0-h))/(2*h)-ref) for h in hs])
print(f"scan center min: h={hs[errs.argmin()]:.6e}, error={errs.min():.6e}")

复合梯形：f=x^2, [0,1]
N  T_N  |I-T_N|  ratio
   4 0.343750000000000 1.041667e-02 
   8 0.335937500000000 2.604167e-03 4.000000
  16 0.333984375000000 6.510417e-04 4.000000
  32 0.333496093750000 1.627604e-04 4.000000
  64 0.333374023437500 4.069010e-05 4.000000
 128 0.333343505859375 1.017253e-05 4.000000

Simpson 三种写法（f=x^2, [0,1], 真值 1/3）
  H/6, midpoint u+H/2        = 0.333333333333333
  (H/2)/6, node u+H/2        = 0.166666666666667
  H/6, node u+H (A1 原式)    = 0.833333333333333

h=0.25: h^(3/6)=0.5; h^3/6=0.00260416666667; ratio=192

[0,1] N=8 error=0.00260416666667; [0,10] N=8 error=2.60416666667; ratio=1000.000000

F'(0.7) = 3.249018709524647
center h=0.1: D=3.241716680946303, D-F'=-7.302028578344e-03, |D-F'|=7.302028578344e-03
forward h=0.1: D=3.383577742132249, D-F'=+1.345590326076e-01
A1 写的 3.2563 = F' + |D-F'| = 3.2563207381
h^2/6 * F'''(0.7) = -7.267710195322e-03；与 D-F' 同号，说明正确符号是 +

A2 的 16 档中心差分扫描（* 为 A2 表中选录的 7 档）
* h=1e-01  |D-F'|=7.302029e-03
* h=1e-02  |D-F'|=7.268056e-05

**读数结论。** 梯形闭式与 A1/A2 一致；A1 的 Simpson 原式在两种读法下都不等于 $1/3$，是确认错误（B1 的 `B-2.3-01` 结论成立）；A1 的 `h^{3/6}` 应写 `h^3/6`（B1 的 `B-2.3-04` 成立）；A1 的 `[0,1]→[0,10]` 误差倍数应为 1000 而非 100（B1 的 `B-2.3-02` 成立）。中心差分表值 `3.2563` 应为 `3.2417166809463027`，误差符号为负；`3.2563` 恰好是 $F'+|D-F'|$，说明不是随机笔误，而是把修正项的符号写反。A2 的 7 行表值全部正确，只是 16 档扫描的选录没有标注；这是展示问题，不是数学错误。

## 2.4 神经网络逼近与 PINN 求导

### 独立推导要点

单隐层 tanh 网络

$$
u_\theta(x)=c+\sum_{j=1}^{q}a_j\tanh(w_jx+b_j).
$$

记 $t_j=\tanh(z_j)$、$s_j=1-t_j^2$、$z_j=w_jx+b_j$，则

$$
u_\theta'(x)=\sum_j a_jw_js_j,\qquad
u_\theta''(x)=-2\sum_j a_jw_j^2t_js_j.
$$

参数导数为

$$
\frac{\partial u}{\partial c}=1,\quad
\frac{\partial u}{\partial a_j}=t_j,\quad
\frac{\partial u}{\partial w_j}=a_js_jx,\quad
\frac{\partial u}{\partial b_j}=a_js_j.
$$

ReLU 网络是分段仿射函数：段内二阶导数为 0，折点处经典二阶导数不存在；框架在折点返回的 0 是约定，不是数学结论。因此强形式二阶 PDE 的残差需要 $C^2$ 激活（如 tanh），或者改用弱形式。

高频反例：$v_k(x)=\sin(\pi x)+\frac1k\sin(kx)$ 满足

$$
\|v_k-u_*\|_\infty=\frac1k,\qquad
|v_k'(0)-u_*'(0)|=1,\qquad
|v_k''(\pi/(2k))-u_*''(\pi/(2k))|=k.
$$

函数值误差可以趋于 0，而二阶导数误差反而增大；这证明“函数拟合好”不能推出“导数拟合好”。

In [5]:
rng = np.random.default_rng(2026)
q = 5
a = rng.normal(size=q); w = rng.normal(size=q); b = rng.normal(size=q); c = 0.3

def net(x):
    t = np.tanh(w*x+b); return c + np.sum(a*t)
def net_d1(x):
    t = np.tanh(w*x+b); s = 1-t*t; return np.sum(a*w*s)
def net_d2(x):
    t = np.tanh(w*x+b); s = 1-t*t; return -2*np.sum(a*w*w*t*s)

x0 = 0.37; h = 1e-5
fd1 = (net(x0+h)-net(x0-h))/(2*h)
fd2 = (net(x0+h)-2*net(x0)+net(x0-h))/h**2
print("tanh 网络：解析导数 vs 中心差分")
print(f"  u' analytic={net_d1(x0):.15g}, fd={fd1:.15g}, diff={abs(net_d1(x0)-fd1):.3e}")
print(f"  u'' analytic={net_d2(x0):.15g}, fd={fd2:.15g}, diff={abs(net_d2(x0)-fd2):.3e}")

# 参数梯度：解析 vs 中心差分
def grads(x):
    t = np.tanh(w*x+b); s = 1-t*t
    ga = t
    gw = a*s*x
    gb = a*s
    gc = 1.0
    return np.array([gc, *ga, *gw, *gb])
g_analytic = grads(x0)
g_fd = []
for k in range(1+3*q):
    if k == 0:
        cp = c + h; cm = c - h
        # temporarily evaluate with modified c
        val_p = cp + np.sum(a*np.tanh(w*x0+b)); val_m = cm + np.sum(a*np.tanh(w*x0+b))
        g_fd.append((val_p-val_m)/(2*h))
    else:
        j = (k-1) % q; which = (k-1)//q
        if which == 0:
            ap = a.copy(); am = a.copy(); ap[j]+=h; am[j]-=h
            vp = c+np.sum(ap*np.tanh(w*x0+b)); vm = c+np.sum(am*np.tanh(w*x0+b))
        elif which == 1:
            wp = w.copy(); wm = w.copy(); wp[j]+=h; wm[j]-=h
            vp = c+np.sum(a*np.tanh(wp*x0+b)); vm = c+np.sum(a*np.tanh(wm*x0+b))
        else:
            bp = b.copy(); bm = b.copy(); bp[j]+=h; bm[j]-=h
            vp = c+np.sum(a*np.tanh(w*x0+bp)); vm = c+np.sum(a*np.tanh(w*x0+bm))
        g_fd.append((vp-vm)/(2*h))
print(f"  parameter gradient max diff = {np.max(np.abs(g_analytic-np.array(g_fd))):.3e}")

# ReLU 的折点行为
relu = lambda x: np.maximum(0.0, x)
print("\nReLU 在折点附近")
for xx in (-0.5, 0.0, 0.5):
    hp = 1e-6
    d = (relu(xx+hp)-relu(xx-hp))/(2*hp)
    print(f"  x={xx:+.1f}: value={relu(xx):.6g}, central diff={d:.6g} (one-sided slopes differ at the kink)")

# 高频反例
print("\n高频反例 v_k=sin(pi x)+(1/k)sin(kx)")
for k in (10, 100, 1000):
    x = np.pi/(2*k)
    ferr = 1.0/k
    d1err = abs(1.0)
    d2err = abs(k)
    print(f"  k={k:4d}: function={ferr:.6e}, first={d1err:.6e}, second={d2err:.6e}")

# A2 的两隐藏层参数计数
print("\n1-32-32-1 参数数 =", (1*32+32)+(32*32+32)+(32*1+1))

# Poisson 残差：u=sin(pi x), -u''=pi^2 sin(pi x)
xg = np.linspace(-1,1,2001)
u = np.sin(np.pi*xg); uxx = -np.pi**2*np.sin(np.pi*xg)
r = -uxx - np.pi**2*np.sin(np.pi*xg)
print("Poisson 残差 max|r| =", np.max(np.abs(r)))

tanh 网络：解析导数 vs 中心差分
  u' analytic=-0.98087830990196, fd=-0.980878309908562, diff=6.602e-12
  u'' analytic=0.0837667726347836, fd=0.083766327207968, diff=4.454e-07
  parameter gradient max diff = 6.466e-11

ReLU 在折点附近
  x=-0.5: value=0, central diff=0 (one-sided slopes differ at the kink)
  x=+0.0: value=0, central diff=0.5 (one-sided slopes differ at the kink)
  x=+0.5: value=0.5, central diff=1 (one-sided slopes differ at the kink)

高频反例 v_k=sin(pi x)+(1/k)sin(kx)
  k=  10: function=1.000000e-01, first=1.000000e+00, second=1.000000e+01
  k= 100: function=1.000000e-02, first=1.000000e+00, second=1.000000e+02
  k=1000: function=1.000000e-03, first=1.000000e+00, second=1.000000e+03

1-32-32-1 参数数 = 1153
Poisson 残差 max|r| = 0.0


**读数结论。** A1 2.5 与 A2 2.4 的链式法则、参数导数表、ReLU 光滑性说明、高频反例均与独立推导一致；未发现数学错误。A2 的 1153 个参数计数正确。A1 把“网络逼近与求导”单列为 2.5，而本章模板是四节 8 学时；这是结构问题，不是公式错误，但会影响 D/E 整合与学时安排。

## 复核结论摘要

| 编号 | 位置 | 问题 | 判断 | 严重度 | 建议 |
| --- | --- | --- | --- | --- | --- |
| CH02-G9-C001 | A1 §2.4 表 | 中心差分值 `3.2563`，误差符号反 | 已确认数值错误；应为 `3.2417166809463027`，误差 `-7.302e-3` | P2（结论不变，数值/符号必须改） | 改表值；保留绝对误差并注明方向 |
| CH02-G9-C002 | A1 §2.3 扩展框 | Simpson 公式 `h/6` 与 `f(u+h)` 混用 | 已确认；半宽读应 `h/3`，全宽读应 `u+H/2` | P1（公式会教错） | 改写为 `H/6[f(u)+4f(u+H/2)+f(v)]` 或 `h/3[f(u)+4f(u+h)+f(v)]` |
| CH02-G9-C003 | A1 题解 4.2 | 中心差分误差符号 `-h²/6` | 已确认；应为 `+h²/6` | P2（与正文矛盾） | 改为 `D-F'=+h²/6 F'''` |
| CH02-G9-C004 | A1 §2.3 | `h^{3/6}` 排版 | 已确认；应为 `h^3/6` | P2（量纲/阶错误） | 三处统一修正 |
| CH02-G9-C005 | A1 §2.3 边界 | `[0,1]→[0,10]` 误差 100 倍 | 已确认；应为 1000 倍 | P1（结论错误） | 改为 `(b-a)h²` 同步放大 |
| CH02-G9-C006 | A1 2.4.2 | `|f|≈3.25` | 已确认；`|F(0.7)|≈2.484`，3.25 是 `|F'|` | P3（最优步长仍同量级） | 改符号或改用 `|F'|` |
| CH02-G9-C007 | A1/A2 2.1 | `n` 定义不同 | 已确认；A1 是次数，A2 是节点数 | P1（跨稿比较/教学） | 加“节点数=次数+1”列或统一 |
| CH02-G9-C008 | A1 全章 | 五节 10 学时 vs 模板四节 8 学时 | 已确认；结构不合模板 | P1（整合阻断） | 合并 2.4+2.5 或由 D/E 明确改学时 |
| CH02-G9-C009 | A2 §2.2 | 400 点测试含 8 个训练点 | 已确认；结论稳健但“独立”不严格 | P2（方法披露） | 排除重合点或改称“含重合点的测试网格” |
| CH02-G9-C010 | A2 §2.3 表 | 7 行 vs 16 档扫描 | 已确认；表值全对，未标“选录” | P3（展示问题，不是数学错误） | 加“选录”或补全 16 行 |
| CH02-G9-C011 | B1 `CH02-G12-B001` | 把 A2 表 7 行判 P1 | 不成立为“数学错误”；应降为 P3 | 严重度校准 | 保留问题，改分级与措辞 |
| CH02-G9-C012 | B2（G22） | 无报告/验证文件 | 证据不足，不能复核 | 待复核 | 等 B2 提交后补审 |